# 第27章　アンサンブルとTTAを実装する ― 最後のひと押し**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## 27.1　交差検証アンサンブル

In [ ]:
import torch, globclass Ensemble:    def __init__(self, ckpt_paths, device=None):        assert len(ckpt_paths) > 0, f"チェックポイントが見つかりません: {ckpt_paths}"  # globの空振りをここで止める        device = device or ("cuda" if torch.cuda.is_available() else "cpu")        self.device = device        self.models = []        for p in ckpt_paths:            m = build_model().to(device)            m.load_state_dict(torch.load(p, map_location=device)["model"])            m.eval()            self.models.append(m)    @torch.no_grad()    def predict(self, x):        probs = [m(x).softmax(1) for m in self.models]   # 各モデルの確率        return torch.stack(probs).mean(0)                # 平均で統合ens = Ensemble(sorted(glob.glob("fold_*/best.pth")))

## 27.2　必要賛成数で運用点を選ぶ

In [ ]:
@torch.no_grad()def majority_vote(models, x, k=4):    votes = sum((m(x).argmax(1) == 1).int() for m in models)   # 陽性と言った数    return (votes >= k).long()                                  # k個以上で陽性# k を下げれば感度↑・過検出↑、上げれば逆（感度と特異度のトレードオフ）

## 27.3　OR（和集合）とAND（積集合）― タスクで「統合」の形が変わる

In [ ]:
@torch.no_grad()def union_masks(models, x):                      # セグメの和集合（OR）    masks = [(m(x).argmax(1) == 1) for m in models]    return torch.stack(masks).any(0).long()      # 一つでも病変ならTrue（見逃さない）    # torch.stack(masks).all(0) にすれば AND（全一致）＝特異度重視

In [ ]:
@torch.no_grad()def union_multilabel(models, x, thr=0.5):        # 多ラベルの所見ごとOR    preds = [(m(x).sigmoid() > thr) for m in models]   # 各所見の有無（独立）    return torch.stack(preds).any(0).long()            # 所見ごとに「1個でもあり」なら陽性

## 27.4　推論時データ拡張（TTA）

In [ ]:
@torch.no_grad()def tta_predict(model, x):    preds = []    # TTAで使ってよいのは、学習時に許した変換だけ。頭尾の反転は25.3のとおり使わない。    # 腹部CTなら反転なしに絞り、左右反転を許した部位でのみ該当軸を足す。    for axes in [()]:                                # 学習時に許した変換に合わせる        xf = torch.flip(x, dims=axes) if axes else x        out = model(xf).softmax(1)        out = torch.flip(out, dims=axes) if axes else out   # 出力を元に戻す        preds.append(out)    return torch.stack(preds).mean(0)                # 平均で安定化